# 04주차 · 문맥 임베딩과 Transformer

**임베딩 기반 데이터 과학 — 국립목포대학교 컴퓨터학부 4학년**

이 Notebook은 *Hands-On Large Language Models* 공개 저장소의 개념과 실습 흐름을
한국어 수업에 맞게 새로 구성한 파생 강의자료입니다. 원본은 Apache License 2.0을
따르며, 출처와 변경 사항은 `SOURCE_AND_LICENSE.md`에 기록했습니다.

- 원본: https://github.com/HandsOnLLM/Hands-On-Large-Language-Models
- 기준 커밋: `ea3390819997999a51983677b80b3aac4dc50ada`
- 권장 환경: Google Colab 또는 Python 3.11+


## 학습목표

- 토큰 임베딩과 문맥 임베딩을 구분한다.
- Scaled dot-product attention을 행렬로 계산한다.
- attention mask와 padding을 제외한 pooling을 계산한다.
- 사전학습 모델의 hidden state 모양을 해석한다.


In [ ]:
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

def cosine(a, b):
    a, b = np.asarray(a, dtype=float), np.asarray(b, dtype=float)
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(a @ b / denom) if denom else 0.0


## Self-Attention

$$Attention(Q,K,V)=softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$


In [ ]:
X = np.array([[1., 0.], [0.8, 0.2], [0., 1.]])
Q = K = V = X
logits = Q @ K.T / np.sqrt(K.shape[1])
exp = np.exp(logits - logits.max(axis=1, keepdims=True))
weights = exp / exp.sum(axis=1, keepdims=True)
contextual = weights @ V
print("Attention weights:\n", np.round(weights, 3))
print("Contextual vectors:\n", np.round(contextual, 3))
assert np.allclose(weights.sum(axis=1), 1.0)


## Mask와 pooling을 수치로 비교하기

`mask=0`인 padding 위치는 attention과 문장 평균에서 제외해야 한다. 아래 예제는 같은 hidden state를 단순 평균한 결과와 실제 토큰만 평균한 결과가 어떻게 달라지는지 보여준다.


In [ ]:
key_mask = np.array([[1, 1, 0]], dtype=bool)  # 마지막 key는 padding
masked_logits = np.where(key_mask, logits, -1e9)
masked_exp = np.exp(masked_logits - masked_logits.max(axis=1, keepdims=True))
masked_weights = masked_exp / masked_exp.sum(axis=1, keepdims=True)
print("Masked attention weights:\n", np.round(masked_weights, 3))
assert np.allclose(masked_weights[:, 2], 0.0)
assert np.allclose(masked_weights.sum(axis=1), 1.0)

hidden_states = np.array([[1.0, 0.0], [0.8, 0.2], [9.0, 9.0]])
token_mask = np.array([1, 1, 0], dtype=float)
plain_mean = hidden_states.mean(axis=0)
masked_mean = (hidden_states * token_mask[:, None]).sum(axis=0) / token_mask.sum()
pd.DataFrame([plain_mean, masked_mean], index=["padding 포함 평균", "mask 적용 평균"], columns=["dim1", "dim2"]).round(3)


## 선택 실습: 한국어 BERT

아래 셀은 인터넷과 `transformers`, `torch`가 필요하다. 실행이 어려우면 위 행렬 실습만으로 핵심 개념을 평가할 수 있다.


In [ ]:
RUN_TRANSFORMER = False  # Colab에서 True로 변경
if RUN_TRANSFORMER:
    import torch
    from transformers import AutoTokenizer, AutoModel
    model_name = "klue/bert-base"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    sentences = ["나는 은행에서 돈을 찾았다", "나는 강둑 은행에 앉았다"]
    batch = tokenizer(sentences, padding=True, return_tensors="pt")
    with torch.no_grad():
        output = model(**batch).last_hidden_state
    print("[batch, tokens, hidden] =", tuple(output.shape))
    mask = batch["attention_mask"].unsqueeze(-1)
    sentence_embeddings = (output * mask).sum(dim=1) / mask.sum(dim=1).clamp(min=1)
    print("[batch, hidden] =", tuple(sentence_embeddings.shape))
else:
    print("선택 실습을 건너뜁니다. RUN_TRANSFORMER=True로 변경하면 실행됩니다.")


## 학생 활동

- Attention 가중치 각 행의 합이 1인지 확인하라.
- padding 벡터 값을 바꾸고 단순 평균과 mask 평균 중 어느 결과만 유지되는지 확인하라.
- 입력 벡터 하나를 크게 바꾸고 모든 출력 벡터에 미치는 영향을 설명하라.
- BERT 선택 실습을 수행했다면 동형이의어가 포함된 한국어 문장 두 쌍을 추가하라.


---
## 학습 기록과 생성형 AI 사용 내역

다음 항목을 자신의 말로 작성하세요.

1. 이번 실습에서 가장 중요한 결과는 무엇인가?
2. 결과를 뒷받침하는 수치 또는 그래프는 무엇인가?
3. 실패하거나 예상과 달랐던 부분은 무엇인가?
4. 생성형 AI를 사용했다면 프롬프트, 채택·거부한 제안, 직접 검증한 내용을 기록하라.
